# 02 — Augmentation des données

Augmente les images du set d'**entraînement uniquement** (jamais val/test, pour éviter la fuite de données), avec une intensité inversement proportionnelle à la taille de la classe afin de rééquilibrer le dataset.

**Transformations utilisées** (conformes au cahier des charges pour images thermiques) :
- rotation légère (+/- 10°)
- zoom léger (0.9x – 1.1x)
- translation légère
- variation de contraste
- flip horizontal (désactivable si le sens physique de l'image compte)

**Volontairement pas utilisé** : bruit, altération de couleur (images thermiques = une seule "couleur" informative, le bruit dégraderait le signal).

**Pré-requis** : avoir exécuté `01_prepare_data.ipynb` (nécessite `data/processed/splits.csv`).


In [7]:
import csv
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
from PIL import Image, ImageEnhance

## Configuration

In [8]:
SPLITS_CSV = Path("data/processed/splits.csv")
OUT_DIR = Path("data/processed/augmented")

# Nombre cible d'images par classe après augmentation.
# Laisser à None pour rééquilibrage complet (= taille de la plus grande classe).
TARGET_PER_CLASS = None

# Mettre à True SEULEMENT si le flip horizontal ne change pas le sens
# physique de l'image (ex : position du défaut / de la caméra).
ALLOW_HFLIP = False

SEED = 42

## Fonctions

In [9]:
def load_split_train(splits_csv: Path):
    by_class = defaultdict(list)
    with open(splits_csv, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row["split"] == "train":
                by_class[row["class"]].append(Path(row["filepath"]))
    return by_class

In [10]:
def augment_image(img: Image.Image, allow_hflip: bool, rng: random.Random) -> Image.Image:
    """Applique une combinaison aléatoire de transformations légères."""
    out = img.copy()

    # Rotation légère
    angle = rng.uniform(-10, 10)
    out = out.rotate(angle, resample=Image.BILINEAR, expand=False)

    # Zoom léger (crop centré + resize)
    zoom = rng.uniform(0.9, 1.1)
    w, h = out.size
    if zoom > 1.0:
        # zoom in : crop puis resize
        new_w, new_h = int(w / zoom), int(h / zoom)
        left = (w - new_w) // 2
        top = (h - new_h) // 2
        out = out.crop((left, top, left + new_w, top + new_h)).resize((w, h), Image.BILINEAR)
    else:
        # zoom out : resize plus petit puis pad
        new_w, new_h = int(w * zoom), int(h * zoom)
        resized = out.resize((new_w, new_h), Image.BILINEAR)
        canvas = Image.new(out.mode, (w, h))
        canvas.paste(resized, ((w - new_w) // 2, (h - new_h) // 2))
        out = canvas

    # Translation légère
    max_shift = 0.05
    dx = int(rng.uniform(-max_shift, max_shift) * w)
    dy = int(rng.uniform(-max_shift, max_shift) * h)
    out = out.transform(out.size, Image.AFFINE, (1, 0, dx, 0, 1, dy))

    # Variation de contraste
    contrast_factor = rng.uniform(0.85, 1.15)
    out = ImageEnhance.Contrast(out).enhance(contrast_factor)

    # Flip horizontal (seulement si autorisé - dépend du sens physique du moteur)
    if allow_hflip and rng.random() < 0.5:
        out = out.transpose(Image.FLIP_LEFT_RIGHT)

    return out

## Exécution

In [11]:
rng = random.Random(SEED)

if not SPLITS_CSV.exists():
    print(f"Fichier introuvable : {SPLITS_CSV}. Lance d'abord 01_prepare_data.ipynb.")
    by_class = {}
else:
    by_class = load_split_train(SPLITS_CSV)

if not by_class:
    print("Aucune image trouvée dans le split 'train'. "
          "Lance d'abord 01_prepare_data.ipynb avec les données réelles.")
else:
    max_count = max(len(v) for v in by_class.values())
    target = TARGET_PER_CLASS or max_count

    OUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Cible par classe : {target} images (avant : {[(c, len(v)) for c, v in by_class.items()]})")

    for cls, files in by_class.items():
        cls_out = OUT_DIR / cls
        cls_out.mkdir(parents=True, exist_ok=True)

        # Copier les originaux
        for i, fp in enumerate(files):
            img = Image.open(fp)
            img.save(cls_out / f"orig_{i:04d}{fp.suffix}")

        n_needed = max(0, target - len(files))
        print(f"  {cls}: {len(files)} originales -> génération de {n_needed} images augmentées")

        for i in range(n_needed):
            src = files[i % len(files)]
            img = Image.open(src)
            aug = augment_image(img, ALLOW_HFLIP, rng)
            aug.save(cls_out / f"aug_{i:04d}{src.suffix}")

    print(f"\nTerminé. Images augmentées écrites dans : {OUT_DIR.resolve()}")
    print("Rappel : val/ et test/ ne sont PAS augmentés (utiliser les originaux du split).")

Cible par classe : 78 images (avant : [('HFL', 78), ('HML', 57), ('HNL', 63), ('CFL', 34), ('CML', 55), ('CNL', 53)])
  HFL: 78 originales -> génération de 0 images augmentées
  HML: 57 originales -> génération de 21 images augmentées
  HNL: 63 originales -> génération de 15 images augmentées
  CFL: 34 originales -> génération de 44 images augmentées
  CML: 55 originales -> génération de 23 images augmentées
  CNL: 53 originales -> génération de 25 images augmentées

Terminé. Images augmentées écrites dans : C:\Users\itsab\Diagnostic_par_classification\data\processed\augmented
Rappel : val/ et test/ ne sont PAS augmentés (utiliser les originaux du split).
